In [1]:
!pip install -q groq python-dotenv pydantic ollama


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import time
import ollama
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel

load_dotenv()
cliente_groq = Groq()

MODELO_GROQ = "qwen/qwen3.8-27b"
MODELO_OLLAMA = "qwen3:8b"

BACKEND_ACTUAL = "ollama"

In [3]:
class Metadatos(BaseModel):
    categoria: str
    nivel_urgencia: str

class Analisis(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: str

class AccionRespuesta(BaseModel):
    codigo_accion_sistema: str
    mensaje_al_cliente: str

class SalidaMaestra(BaseModel):
    metadatos_clasificacion: Metadatos
    analisis_interno: Analisis
    accion_y_respuesta: AccionRespuesta

In [4]:
def llamar_llm(system_prompt: str, input_data: dict) -> dict:
    global BACKEND_ACTUAL

    mensajes = [
        {
            "role": "system",
            "content": f"{system_prompt}\n\nDebes responder ÚNICAMENTE con un objeto JSON válido."
        },
        {
            "role": "user",
            "content": json.dumps(input_data)
        }
    ]

    if BACKEND_ACTUAL == "groq":
        chat_completion = cliente_groq.chat.completions.create(
            messages=mensajes,
            model=MODELO_GROQ,
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        return json.loads(chat_completion.choices[0].message.content)

    elif BACKEND_ACTUAL == "ollama":
        respuesta = ollama.chat(
            model=MODELO_OLLAMA,
            messages=mensajes,
            format="json",
            options={"temperature": 0.0}
        )
        return json.loads(respuesta['message']['content'])

In [5]:
def procesar_mensaje(mensaje_usuario: str) -> dict:

    try:
        prompt_urgencia = open("prompt_urgencia.md", "r", encoding="utf-8").read()
        prompt_categoria = open("prompt_categoria.md", "r", encoding="utf-8").read()
        prompt_accion = open("prompt_accion.md", "r", encoding="utf-8").read()
        prompt_texto = open("prompt_texto.md", "r", encoding="utf-8").read()
        prompt_maestro = open("prompt_maestro.md", "r", encoding="utf-8").read()
    except FileNotFoundError as e:
        raise RuntimeError(f"Error al leer los archivos de prompt: {e}")

    def manejar_pausa():
        if BACKEND_ACTUAL == "groq":
            time.sleep(15)

    print(f"[{BACKEND_ACTUAL.upper()}] Procesando...")

    out_urgencia = llamar_llm(prompt_urgencia, {"mensaje_usuario": mensaje_usuario})
    nivel_urgencia = out_urgencia.get("nivel_urgencia", "")
    manejar_pausa()

    out_categoria = llamar_llm(prompt_categoria, {"mensaje_usuario": mensaje_usuario})
    categoria = out_categoria.get("categoria", "")
    manejar_pausa()

    out_accion = llamar_llm(
        prompt_accion,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria}
    )
    cadena_de_pensamiento = out_accion.get("cadena_de_pensamiento", "")
    politica_aplicada = out_accion.get("politica_aplicada", "")
    codigo_accion_sistema = out_accion.get("codigo_accion_sistema", "")
    manejar_pausa()

    out_texto = llamar_llm(
        prompt_texto,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria, "politica_aplicada": politica_aplicada}
    )
    mensaje_al_cliente = out_texto.get("mensaje_al_cliente", "")
    manejar_pausa()

    out_maestro = llamar_llm(
        prompt_maestro,
        {
            "categoria": categoria,
            "nivel_urgencia": nivel_urgencia,
            "cadena_de_pensamiento": cadena_de_pensamiento,
            "politica_aplicada": politica_aplicada,
            "codigo_accion_sistema": codigo_accion_sistema,
            "mensaje_al_cliente": mensaje_al_cliente
        }
    )

    salida_validada = SalidaMaestra(**out_maestro)
    return salida_validada.model_dump()

In [ ]:
# EJEMPLO DE USO 1
if __name__ == "__main__":
    mensaje = "Hola, compré una cafetera hace dos meses y ayer dejó de prender. Además me cobraron dos veces el envío en la tarjeta. Quiero mi plata ya o los demando."
    try:
        resultado = procesar_mensaje(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

[OLLAMA] Procesando...


In [ ]:
# EJEMPLO DE USO 2
if __name__ == "__main__":
    mensaje = "El estado de mi pedido dice 'entregado' hace 3 días, pero yo estaba de viaje y mis cámaras muestran que nadie vino. Para colmo, en el resumen de la tarjeta me aparece un recargo del 15% que no reconozco. Ayuda urgente."
    try:
        resultado = procesar_mensaje(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

In [ ]:
# EJEMPLO DE USO 3
if __name__ == "__main__":
    mensaje = "Compré 4 sillas y una mesa. Llegaron hoy, pero 2 sillas tienen las patas astilladas y la mesa no trajo los tornillos para armarla. Tengo una cena mañana y necesito esto resuelto hoy mismo."
    try:
        resultado = procesar_mensaje(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

In [ ]:
# EJEMPLO DE USO 4
if __name__ == "__main__":
    mensaje = "Estoy intentando descargar mi factura A desde la plataforma pero el PDF da error. Además, la caja del mouse que compré vino aplastada, aunque el mouse funciona bien. Lo que más me importa es la factura para el contador hoy."
    try:
        resultado = procesar_mensaje(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

In [ ]:
# EJEMPLO DE USO 5
if __name__ == "__main__":
    mensaje = "El repartidor me acaba de llamar pidiendo que le pague un 'extra por zona peligrosa' en efectivo. Me negué y se fue con mi paquete (una notebook de alta gama). Ya pagué el envío premium con la tarjeta. ¡Exijo que me lo traigan ya y sancionen al repartidor!"
    try:
        resultado = procesar_mensaje(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")